In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds


In [ ]:
(ds_train, ds_val, ds_test), ds_info = tfds.load(
    'malaria',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True
)

print("Dataset loaded with separate training, validation, and test splits.")

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10

class_names = ['Parasitized', 'Uninfected']
num_classes = 2


In [ ]:
def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0
    return image, label

In [ ]:
train_ds = (
    ds_train.map(preprocess)
    .shuffle(10000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = ds_val.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = ds_test.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

In [ ]:
model = models.Sequential([

    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    data_augmentation,

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
print("CNN Test Accuracy:", test_accuracy)
print("CNN Test Loss:", test_loss)

In [ ]:
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.show()

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np

cnn_y_true = []
cnn_y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    cnn_y_true.extend(labels.numpy())
    cnn_y_pred.extend(np.argmax(predictions, axis=1))

cnn_cm = confusion_matrix(cnn_y_true, cnn_y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cnn_cm,
    display_labels=['Parasitized', 'Uninfected']
)
disp.plot()
plt.title("CNN Test Confusion Matrix")
plt.show()

In [ ]:
model.save("malaria_cnn_model.h5")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

for images, labels in test_ds.take(1):
    plt.figure(figsize=(15, 10))
    for i in range(min(5, len(images))):
        plt.subplot(1, 5, i + 1)
        plt.imshow(images[i])
        plt.axis("off")
        img_array = np.expand_dims(images[i], axis=0)
        prediction = model.predict(img_array, verbose=0)
        predicted_class = class_names[np.argmax(prediction)]
        true_class = class_names[labels[i].numpy()]
        confidence = np.max(prediction)
        plt.title(f"True: {true_class}\nPred: {predicted_class}\n{confidence:.2f}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Install project dependencies before running:
# pip install -r requirements.txt

In [ ]:
import tensorflow_datasets as tfds
import os
import shutil
import cv2

# Re-create folders so old exported images cannot leak into a new split.
shutil.rmtree("malaria_dataset", ignore_errors=True)

(ds_train_yolo, ds_val_yolo, ds_test_yolo), ds_info_yolo = tfds.load(
    'malaria',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    as_supervised=True,
    with_info=True
)

classes = ['Parasitized', 'Uninfected']
splits = {
    "train": ds_train_yolo,
    "val": ds_val_yolo,
    "test": ds_test_yolo
}

for split_name, dataset in splits.items():
    for class_name in classes:
        os.makedirs(f"malaria_dataset/{split_name}/{class_name}", exist_ok=True)

    for i, (image, label) in enumerate(dataset):
        class_name = classes[int(label.numpy())]
        image_array = image.numpy()
        output_path = f"malaria_dataset/{split_name}/{class_name}/{i}.jpg"
        cv2.imwrite(output_path, cv2.cvtColor(image_array, cv2.COLOR_RGB2BGR))

print("YOLO dataset exported with train, val, and test splits.")

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolo11n-cls.pt")

yolo_results = yolo_model.train(
    data="malaria_dataset",
    epochs=10,
    imgsz=128,
    batch=32
)

In [ ]:
# Evaluate only on the held-out test split.
yolo_test_metrics = yolo_model.val(data="malaria_dataset", split="test")
print("YOLO Test Top-1 Accuracy:", yolo_test_metrics.top1)
print("YOLO Test Top-5 Accuracy:", yolo_test_metrics.top5)

In [ ]:
cnn_test_loss, cnn_test_accuracy = model.evaluate(test_ds, verbose=1)
print("CNN Test Accuracy:", cnn_test_accuracy)
print("CNN Test Loss:", cnn_test_loss)

In [ ]:
print("YOLO Test Top-1 Accuracy:", yolo_test_metrics.top1)
print("YOLO Test Top-5 Accuracy:", yolo_test_metrics.top5)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import glob
import os

yolo_y_true = []
yolo_y_pred = []
class_names = ['Parasitized', 'Uninfected']
test_image_paths = sorted(glob.glob("malaria_dataset/test/*/*.jpg"))

for img_path in test_image_paths:
    true_class_name = os.path.basename(os.path.dirname(img_path))
    yolo_y_true.append(class_names.index(true_class_name))
    result = yolo_model(img_path, verbose=False)
    yolo_y_pred.append(result[0].probs.top1)

yolo_cm = confusion_matrix(yolo_y_true, yolo_y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=yolo_cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax)
plt.title("YOLO Test Confusion Matrix")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

for images, labels in test_ds.take(1):
    plt.figure(figsize=(15, 10))
    for i in range(min(5, len(images))):
        plt.subplot(1, 5, i + 1)
        plt.imshow(images[i])
        plt.axis("off")
        image_array = (images[i].numpy() * 255).astype("uint8")
        result = yolo_model(image_array, verbose=False)
        predicted_class = class_names[result[0].probs.top1]
        true_class = class_names[labels[i].numpy()]
        confidence = float(result[0].probs.top1conf)
        plt.title(f"True: {true_class}\nPred: {predicted_class}\n{confidence:.2f}")
    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    yolo_y_true,
    yolo_y_pred,
    target_names=['Parasitized', 'Uninfected']
))

In [ ]:
import matplotlib.pyplot as plt

models = ["CNN", "YOLO"]
test_accuracies = [
    cnn_test_accuracy * 100,
    yolo_test_metrics.top1 * 100
]

plt.bar(models, test_accuracies)
plt.title("CNN vs YOLO Test Accuracy")
plt.xlabel("Model")
plt.ylabel("Accuracy (%)")
plt.ylim(0, 100)
plt.show()